# CityPilot IA · Cuaderno de validación

**Objetivo:** usar el contenedor Jupyter como **cliente independiente** de la web y PostgreSQL. Completa las celdas, ejecútalas y guarda los resultados en el repositorio. No es un notebook decorativo: los 4 controles son obligatorios.

> Dentro del contenedor, la URL de Flask es `http://web:5000`; `localhost:5000` apuntaría al propio Jupyter. Los datos de los fixtures son de demostración; no confundirlos con respuestas vivas.

In [ ]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine, text

WEB = os.environ.get("WEB_BASE_URL", "http://web:5000")
DB = os.environ["DATABASE_URL"]
print("API:", WEB)
print("Base configurada:", bool(DB))

## 0. Salud de la arquitectura (pista resuelta)
Comprueba que Jupyter puede llegar a Flask y que Flask responde sobre la BD. Explica por escrito qué significa cada conexión.

In [ ]:
respuesta = requests.get(f"{WEB}/api/health", timeout=5)
print(respuesta.status_code, respuesta.json())
assert respuesta.status_code == 200

## 1. API de datos públicos (obligatorio)
Con `requests`, recupera **Valencia → coordenadas → clima actual**. No uses lat/lon de memoria como respuesta final: debes sacarlas del JSON de Geocoding. Muestra ciudad, lat, lon, temperatura y precipitación. Comprueba qué haces si `results` está vacío.

En `docs/APIS_Y_CONTRATOS.md` tienes URLs y parámetros. Puedes poner los campos `name`, `count`, `countryCode`, `current`, `timezone` en diccionarios de parámetros.

In [ ]:
geo_url = "https://geocoding-api.open-meteo.com/v1/search"
meteo_url = "https://api.open-meteo.com/v1/forecast"
# TODO(ALUMNADO): petición GET geocoding, extraer latitud/longitud
# TODO(ALUMNADO): petición GET meteorología con esas coordenadas
# TODO(ALUMNADO): imprimir resultado y dejar evidencia


## 2. API de vuestro Flask (obligatorio)
Desde **Jupyter** haz una petición POST JSON a `/api/planificar`. Comprueba: status 200, campos `itinerario`, `meteorologia`, `lugares`, `solicitud_id` y que la ciudad coincide con la enviada. Usa un tipo del conjunto aceptado. Si no tenéis clave Cohere, `LLM_MODE=mock` permite validar transporte y flujo; **no demuestra generación real**.

In [ ]:
payload = {
    "ciudad": "Valencia",
    "tipo_turismo": "cultural",
    "duracion_horas": 4,
    "preferencias": "poco caminar",
}
# TODO(ALUMNADO): requests.post(..., json=payload, timeout=...)
# TODO(ALUMNADO): verificar status, estructura JSON y datos


## 3. PostgreSQL (obligatorio)
Después de crear rutas mediante API, conecta a PostgreSQL desde este notebook. Obtén el número de solicitudes y una tabla `id, fecha, ciudad, tipo_turismo, estado` con hasta cinco filas. Compara con el número y últimas filas que ves en vuestro dashboard.

In [ ]:
engine = create_engine(DB, pool_pre_ping=True)
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM solicitudes")).scalar_one()
print("Filas en PostgreSQL:", n)
# TODO(ALUMNADO): consulta las últimas 5 solicitudes y preséntalas como DataFrame


## 4. Error controlado (obligatorio)
Elige un caso de validación: ciudad vacía, `duracion_horas=99` o `tipo_turismo=espacial`. Haz `POST /api/planificar` y demuestra que la aplicación responde **400** con un mensaje claro. Una respuesta 500 inesperada no supera esta prueba.

In [ ]:
# TODO(ALUMNADO): realiza POST con entrada inválida
# TODO(ALUMNADO): assert del HTTP 400 y lectura del mensaje de error


## 5. Bonus · Analítica desde el notebook
Con `pandas.read_sql_query` y SQLAlchemy, compara las ciudades y tipos más solicitados. Si te interesa, haz aquí una figura de Matplotlib; la figura obligatoria del dashboard **debe seguir saliendo del servidor web y de PostgreSQL**, no copiarse manualmente desde el notebook.

In [ ]:
# BONUS: SELECT ciudad, COUNT(*) ... GROUP BY ciudad ...
# BONUS: pd.read_sql_query(text(...), engine) y una gráfica


### Evidencia final
Guarda el notebook con salidas visibles, sin claves y sin capturas sensibles. En vuestro README, describe brevemente **una conclusión** de los datos almacenados. ¡Cuidado! Las filas `demo=true` son simuladas y se deben identificar como tales.

**Semáforo amarillo:** en tu memoria individual indica si la IA te ayudó a escribir celdas, consultas o asserts; explica qué corregiste y dónde se ve tu prueba. No incluir claves.